# Lab 4: The PyTorch training pipeline

**DSAN-6600 Deep Learning · Fall 2026**

**Due:** Wednesday Sep 30, 11:59 PM ET · **Target time:** about 60 minutes

---

## What this lab is for

In Lab 2 you wrote a forward pass. In Lab 3 you wrote the backward pass, twice.
You now know what `loss.backward()` does, because you have written it.

So this lab does not ask you to write any of that again. PyTorch does it in one
line. What you write instead is **everything around it**, which is where real
training runs are actually won and lost:

1. A `Dataset` that says how to produce one training example, and a
   `DataLoader` that feeds the model fast enough to keep it busy.
2. The training loop itself, which is five lines and has four ways to go wrong.
3. The optimizer, which changes your answer more than you might expect.
4. The checks that tell you whether the run is any good: a sanity check before
   you start, a validation curve while it runs, and a test set you touch once.

This is also the first lab on **real data**. Fashion-MNIST: 70,000 grayscale
images of clothing, 28 by 28 pixels, ten classes. It is a drop-in replacement
for MNIST that is hard enough to be interesting.

## How this lab is graded

- **The notebook is graded for completion, not for correctness.** Submit it,
  with all cells run and outputs visible, and you get the points.
- **You may work together.** Everyone runs the same seed, so your numbers
  should land very close to your neighbor's. They will not be identical to the
  last decimal if one of you is on a GPU and the other is on a CPU, which is
  normal and is not a bug.
- **The understanding is assessed on Quiz 5**, which is the Thursday after the
  due date. Part of that quiz asks you to explain what this lab demonstrated
  and why.
- **Solutions are posted after the deadline.**

Project Check-in 1 is due the same day, so this lab is deliberately shorter
than Lab 3. Nothing here requires you to derive anything.

## Using AI on this lab

Allowed and expected, with disclosure; there is a cell for that at the end. An
agent can write this pipeline in a minute. It cannot sit Thursday's quiz for
you, and it will not notice that your validation set leaked.

## Step 0: Setup

Two things to know before you run this.

**The download.** The first cell fetches Fashion-MNIST into a `data/` folder
next to this notebook, about 82 MB on disk. It takes a few seconds and only
happens once; after that `download=True` finds the files and does nothing.

**How long it runs.** Everything here is sized for a laptop CPU. The notebook
trains ten models end to end and the whole thing takes about a minute of
compute on a desktop, a few minutes on Colab's free tier. You do not need a
GPU, and if you have one the code will use it without any changes.

**`NUM_WORKERS`.** Part 1 measures what background loading processes do for
throughput. On macOS and Linux this just works. On **Windows**, `num_workers`
greater than zero inside a Jupyter notebook is sometimes slow and occasionally
hangs, because Windows starts workers with `spawn` rather than `fork`. If a
cell in Part 1d hangs for more than a minute, interrupt the kernel, set
`NUM_WORKERS = 0` below, and re-run. You will lose the measurement, not the
lesson; the numbers are reprinted in the text.

In [ ]:
import time
import platform

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets

SEED = 6600
torch.manual_seed(SEED)
np.random.seed(SEED)

# Everything in this lab is sized for a CPU. If you have a GPU it will be
# faster, but nothing here needs one.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Set this to 0 if Part 1d misbehaves on Windows. See the note above.
NUM_WORKERS = 0 if platform.system() == "Windows" else 2

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"torch {torch.__version__} on {DEVICE}")
print(f"platform {platform.system()}, NUM_WORKERS = {NUM_WORKERS}")

---

# Part 1 · Dataset and DataLoader

## 1a · The raw data

`torchvision` hands back the images as a single `uint8` tensor and the labels
as a tensor of integers 0 through 9. No normalization, no flattening, no
batching. That is on purpose: those are your decisions, and you are about to
write them down.

In [ ]:
train_raw = datasets.FashionMNIST("data", train=True, download=True)
test_raw = datasets.FashionMNIST("data", train=False, download=True)

CLASSES = train_raw.classes

print(f"train images {tuple(train_raw.data.shape)}  {train_raw.data.dtype}"
      f"  values {int(train_raw.data.min())} to {int(train_raw.data.max())}")
print(f"test  images {tuple(test_raw.data.shape)}")
print(f"labels       {tuple(train_raw.targets.shape)}  {train_raw.targets.dtype}")
print()
for i, c in enumerate(CLASSES):
    print(f"  {i}  {c}")

Look at them before you model them. This is not a formality: half the data bugs
you will ever have are visible in a picture and invisible in a shape.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(11, 3.2))
for ax, i in zip(axes.ravel(), range(16)):
    ax.imshow(train_raw.data[i], cmap="gray_r")
    ax.set_title(CLASSES[train_raw.targets[i]], fontsize=7)
    ax.axis("off")
    ax.grid(False)
plt.tight_layout()
plt.show()

## 1b · Write the Dataset

A `Dataset` is not a container. It is a **recipe for producing one training
example on demand**, and it answers three things:

| Method | What it answers |
|---|---|
| `__init__` | Where the data lives and what I need to keep to reach it |
| `__len__` | How many examples there are |
| `__getitem__(i)` | Here is example `i`, ready for the model |

That third one is the one people underrate. `__getitem__` is where the work
happens: the type conversion, the scaling, the reshape, and on real image
datasets the JPEG decode and the augmentation. It runs **once per example, per
epoch**, which is why Part 1d is about how fast it is.

Your `__getitem__` has three jobs. The images arrive as `uint8` in
$[0, 255]$ shaped $(28, 28)$, and the model in Part 2 wants `float32` in
$[0, 1]$ shaped $(784,)$.

::: {.callout-note}
## Return the label as it came
Do not one-hot encode the label. `CrossEntropyLoss` wants the raw class index,
an integer from 0 to 9. This surprises people every single semester.
:::

In [ ]:
class FashionDataset(Dataset):
    """One example at a time: uint8 (28,28) in -> float32 (784,) in [0,1] out."""

    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        # TODO: how many examples are in this dataset?
        raise NotImplementedError("__len__")

    def __getitem__(self, i):
        # TODO: return (x, y) for example i, where x is a float32 tensor of
        #   shape (784,) with values in [0, 1], and y is the label unchanged.
        #   Three steps on the image: cast to float, scale by 255, flatten.
        #   .float(), / 255.0 and .view(-1) will do it.
        raise NotImplementedError("__getitem__")

## 1c · Split the data, and split it correctly

Three sets, and the rule that matters is **where the validation set comes
from**:

- **Train** is what the optimizer sees.
- **Validation** is what *you* see, every epoch, to decide when to stop and
  which settings to keep.
- **Test** is what nobody sees until the very end.

The validation set is carved out of the **training** set, never out of test. If
you tune against test, your test number stops estimating anything, because you
have been fitting to it by hand. That is the most common form of leakage and it
does not announce itself; the number just comes out too good.

We train on 6,000 examples rather than all 60,000. That is not to save time,
though it does. With 6,000 examples this model overfits clearly within 40
epochs, and Part 4 needs that to be visible.

In [ ]:
TRAIN_N, VAL_N = 6_000, 2_000

# A seeded permutation, so everyone in the class gets the same split.
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(train_raw.data), generator=g)
tr_idx, va_idx = perm[:TRAIN_N], perm[TRAIN_N:TRAIN_N + VAL_N]

train_ds = FashionDataset(train_raw.data[tr_idx], train_raw.targets[tr_idx])
val_ds = FashionDataset(train_raw.data[va_idx], train_raw.targets[va_idx])
test_ds = FashionDataset(test_raw.data, test_raw.targets)

x0, y0 = train_ds[0]
print(f"train {len(train_ds):>6,}   val {len(val_ds):>6,}   test {len(test_ds):>6,}")
print()
print(f"one example: x is {tuple(x0.shape)} {x0.dtype} "
      f"in [{x0.min():.2f}, {x0.max():.2f}]")
print(f"             y is {y0.dtype} = {int(y0)} ({CLASSES[y0]})")

assert x0.shape == (784,) and x0.dtype == torch.float32, "check __getitem__"
assert 0.0 <= float(x0.min()) and float(x0.max()) <= 1.0, "scale to [0, 1]"
print("\nshapes and ranges check out")

## 1d · What the DataLoader is actually for

The `DataLoader` wraps a `Dataset` and does four things: batching, shuffling,
collating into tensors, and **loading in the background while the GPU works**.

The first three are bookkeeping. The fourth is why it exists. If your model
takes 20 ms per batch and your loader takes 40 ms to build one, your expensive
hardware is idle two thirds of the time, and no amount of optimizer tuning
fixes that.

So measure it. Below are two datasets with identical shapes. One is the
in-memory tensor slice you just wrote. The other has a `__getitem__` that does
real work, standing in for decoding a JPEG and augmenting it.

In [ ]:
class SlowDataset(Dataset):
    """Same shapes as FashionDataset, but __getitem__ costs something.

    Stands in for the real case: read a JPEG off disk, decode it, resize,
    random crop, random flip, normalize. That is a few milliseconds per image.
    """

    def __init__(self, n=2048):
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        a = np.random.rand(140, 140)
        for _ in range(9):
            a = np.sqrt(a + 0.5)
        return torch.from_numpy(a[:28, :28].ravel().astype(np.float32)), i % 10


def time_one_pass(ds, num_workers, **kw):
    """Walk the whole dataset once and report images per second."""
    dl = DataLoader(ds, batch_size=128, num_workers=num_workers, **kw)
    t0 = time.perf_counter()
    seen = sum(len(y) for _, y in dl)
    dt = time.perf_counter() - t0
    return dt, seen / dt


worker_counts = sorted({0, NUM_WORKERS, NUM_WORKERS * 2}) if NUM_WORKERS else [0]

print("in-memory tensors, __getitem__ is nearly free")
for nw in worker_counts:
    dt, rate = time_one_pass(train_ds, nw)
    print(f"   num_workers={nw}: {dt:5.2f}s   {rate:>9,.0f} images/sec")

print("\nexpensive __getitem__, like decoding real images")
for nw in worker_counts:
    dt, rate = time_one_pass(SlowDataset(), nw)
    print(f"   num_workers={nw}: {dt:5.2f}s   {rate:>9,.0f} images/sec")

**Read those two blocks against each other.** They point in opposite
directions, and that is the lesson.

On the in-memory dataset, workers make it **slower**. `__getitem__` is a tensor
slice that takes microseconds, so there is no work to parallelize, and you have
added process startup and the cost of shipping every batch back through a pipe.

On the expensive dataset, workers roughly **triple** the throughput, because
now there is real work to overlap and it happens while the main process trains.

The rule is not "always use more workers." It is:

> Workers help exactly when `__getitem__` is slow. Measure before you tune.

Two more settings you will see, and when they matter:

| Setting | What it does | When to use it |
|---|---|---|
| `num_workers=k` | `k` background processes build batches | When `__getitem__` does real work |
| `pin_memory=True` | Staging buffer that copies to GPU faster | Training on a GPU; useless on CPU |
| `persistent_workers=True` | Keep workers alive between epochs | With `num_workers > 0` and short epochs |
| `prefetch_factor=n` | Batches each worker runs ahead | When batch times are uneven |
| `drop_last=True` | Discard the ragged final batch | When a fixed batch size matters |

In [ ]:
# The loaders the rest of the lab uses. Shuffle the training data every epoch;
# never shuffle validation or test, there is no reason to.
#
# The explicit generator is what makes a re-run of a single cell reproduce its
# own output. torch.manual_seed() at the top of the notebook is not enough,
# because any other random call in between moves the global stream.
BATCH_SIZE = 128

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      generator=torch.Generator().manual_seed(SEED))
val_dl = DataLoader(val_ds, batch_size=512)
test_dl = DataLoader(test_ds, batch_size=512)

xb, yb = next(iter(train_dl))
print(f"one batch: x {tuple(xb.shape)}  y {tuple(yb.shape)}")
print(f"batches per epoch: {len(train_dl)}")

---

# Part 2 · The model and the training loop

## 2a · The model

784 inputs, one hidden layer of 256 with ReLU, 10 outputs. About 200,000
parameters, which is four times the number of training examples we gave it.
Remember that when Part 4 overfits.

::: {.callout-important}
## Do not put a softmax on the end
`nn.CrossEntropyLoss` applies `log_softmax` internally. If you add your own
softmax, you apply it twice: the gradients get small, training crawls, and
nothing errors out to tell you. **The last layer emits raw logits.**
:::

In [ ]:
def make_model(p_drop=0.0, seed=SEED):
    """784 -> 256 -> ReLU -> [dropout] -> 10 logits.

    Seeded inside, so every model in this lab starts from identical weights and
    any difference you see downstream is the thing you changed.
    """
    torch.manual_seed(seed)

    # TODO: build the network with nn.Sequential and return it.
    #   nn.Linear(784, 256), then nn.ReLU(),
    #   then nn.Dropout(p_drop) but ONLY if p_drop > 0,
    #   then nn.Linear(256, 10).
    #   Build a list of layers, append conditionally, nn.Sequential(*layers).
    raise NotImplementedError("make_model")


model = make_model().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())

print(model)
print(f"\n{n_params:,} parameters for {TRAIN_N:,} training examples")

## 2b · The training loop

Five lines, in this order, once per batch:

```python
optimizer.zero_grad()      # 1. clear last batch's gradients
loss = loss_fn(model(xb), yb)   # 2 and 3. forward, then score it
loss.backward()            # 4. every .grad in the model gets filled in
optimizer.step()           # 5. every parameter moves
```

Line 1 is the one people leave out, and leaving it out is not an error. PyTorch
**accumulates** into `.grad` rather than overwriting, which is deliberate and
useful. Skip `zero_grad()` and step 5 uses the sum of every gradient since the
beginning of the epoch. The loss goes strange and nothing raises.

You wrote the machinery behind line 4 in Lab 3. It is the topological sort and
the chain rule, on a graph PyTorch built while line 2 ran.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    """One pass over the training data. Returns the mean loss per example."""
    model.train()
    running = 0.0

    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        # TODO: the five lines, in order.
        #   zero the gradients, forward pass, compute loss,
        #   backward, then step. Name the loss `loss` so the line below works.
        raise NotImplementedError("train_one_epoch")

        running += loss.item() * len(yb)

    return running / len(loader.dataset)

## 2c · Evaluation is a different function

This one is provided, because the two lines that matter are easy to describe
and easy to forget.

`model.eval()` switches dropout off and makes batch norm use its running
statistics. Forget it and your validation number is computed with dropout still
firing, so it comes out worse than the model really is.

`torch.no_grad()` tells autograd to stop building the graph. You are not going
to call `.backward()` here, so every saved activation is wasted memory. It also
runs faster.

Note `reduction="sum"` and the division at the end. Averaging the per-batch
averages would silently weight the ragged last batch the same as a full one.

In [ ]:
def evaluate(model, loader):
    """Mean loss and accuracy over a loader. No gradients, no dropout."""
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    total_loss, correct, n = 0.0, 0, 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            total_loss += loss_fn(logits, yb).item()
            correct += (logits.argmax(dim=1) == yb).sum().item()
            n += len(yb)

    return total_loss / n, correct / n


loss, acc = evaluate(make_model().to(DEVICE), val_dl)
print(f"untrained model: loss {loss:.4f}, accuracy {acc:.3f}")
print(f"ten classes, so chance is 0.100 and ln(10) = {np.log(10):.4f}")

**The loss is the check here, not the accuracy.** An untrained model has no
reason to prefer any class, so it scores about $\ln(10) = 2.303$, and that
number is a genuine diagnostic: if your untrained loss is far from 2.303,
something is already wrong. Your labels are misaligned, or your inputs are not
scaled, or you put a softmax on the end. Check it before you spend an hour
training.

The accuracy at initialization is *not* a useful check. An untrained network
usually predicts nearly the same class for everything, so its accuracy is
whatever that one class happens to be worth, which can be far from 10 percent
in either direction.

## 2d · Overfit 100 examples first

**Do this before every real training run.** It takes fifteen seconds and it
separates "my model is not learning" from "my pipeline is broken."

Take 100 examples and train until the model has memorized them. A working
pipeline drives the loss to nearly zero and the accuracy to 100 percent,
because a 200,000-parameter model can absolutely memorize 100 images. If it
cannot, the bug is not your learning rate. Your labels are shuffled relative to
your inputs, or your gradients are not reaching the weights, and you have
learned that in fifteen seconds instead of after an hour of real training.

In [ ]:
tiny_ds = Subset(train_ds, range(100))
tiny_dl = DataLoader(tiny_ds, batch_size=100, shuffle=True,
                     generator=torch.Generator().manual_seed(SEED))

tiny_model = make_model().to(DEVICE)
tiny_opt = torch.optim.Adam(tiny_model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

t0 = time.perf_counter()
for epoch in range(200):
    # TODO: one call to train_one_epoch on the tiny loader. Keep the return
    #   value in `L` so the reporting below works.
    raise NotImplementedError("tiny loop")

    if epoch in (0, 49, 99, 199):
        print(f"  epoch {epoch + 1:>3}   loss {L:.6f}")

_, tiny_acc = evaluate(tiny_model, tiny_dl)
print(f"\nfinal loss {L:.2e}, accuracy {tiny_acc:.3f}, "
      f"in {time.perf_counter() - t0:.1f}s")
print("loss near zero and accuracy 1.000 means the pipeline works")

::: {.callout-note}
## This is a sanity check, not a result
Memorizing 100 examples proves the plumbing works. It says nothing at all about
whether the model generalizes. That is what the validation set is for, and it
is the next cell.
:::

## 2e · The real run

Now the whole thing: 40 epochs on 6,000 examples, measuring validation after
every one. This is the loop you will write for the rest of the semester.

In [ ]:
def fit(model, optimizer, epochs=40, loss_fn=None, scheduler=None, log_every=10):
    """Train for a fixed number of epochs, recording train and validation."""
    loss_fn = loss_fn or nn.CrossEntropyLoss()
    hist = {"train": [], "val": [], "acc": []}

    # Rewind the loader's shuffle so every run in this lab sees the batches in
    # the same order. A generator is stateful: without this, the second call to
    # fit() would get a different shuffle than the first, and any comparison
    # between two runs would include that difference.
    train_dl.generator.manual_seed(SEED)

    t0 = time.perf_counter()
    for epoch in range(epochs):
        tr = train_one_epoch(model, train_dl, optimizer, loss_fn)
        vl, va = evaluate(model, val_dl)
        if scheduler is not None:
            scheduler.step()

        hist["train"].append(tr)
        hist["val"].append(vl)
        hist["acc"].append(va)

        if log_every and (epoch % log_every == 0 or epoch == epochs - 1):
            print(f"  epoch {epoch:>3}   train {tr:.4f}   "
                  f"val {vl:.4f}   acc {va:.4f}")

    hist["seconds"] = time.perf_counter() - t0
    return hist


model = make_model().to(DEVICE)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

print("SGD with momentum, lr=0.1")
hist = fit(model, optimizer)
print(f"\n{hist['seconds']:.1f}s total, "
      f"{hist['seconds'] / len(hist['train']):.2f}s per epoch")

In [ ]:
def plot_history(hists, title):
    """Loss curves on the left, validation accuracy on the right."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    for i, (name, h) in enumerate(hists.items()):
        c = f"C{i}"
        ax1.plot(h["train"], color=c, lw=1.6, label=f"{name}, train")
        ax1.plot(h["val"], color=c, lw=1.6, ls="--", label=f"{name}, val")
        best = int(np.argmin(h["val"]))
        ax1.plot(best, h["val"][best], "o", color=c, ms=7, mfc="white", mew=2)
        ax2.plot(h["acc"], color=c, lw=1.6, label=name)

    ax1.set_xlabel("epoch"); ax1.set_ylabel("cross-entropy")
    ax1.set_title("solid = train, dashed = validation")
    ax1.legend(fontsize=8)
    ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy")
    ax2.set_title("validation accuracy")
    ax2.legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_history({"SGD+momentum": hist}, "One run, 6,000 training examples")

best = int(np.argmin(hist["val"]))
print(f"best validation loss {hist['val'][best]:.4f} at epoch {best}")
print(f"final validation loss {hist['val'][-1]:.4f} at epoch "
      f"{len(hist['val']) - 1}")
print(f"training loss went from {hist['train'][0]:.4f} to {hist['train'][-1]:.4f}")

**Look at the gap between the two curves in the left panel.** The solid
training curve keeps going down. The dashed validation curve stops, turns
around, and drifts back up. The white circle marks its lowest point. Past that
circle, the extra training is buying accuracy on the 6,000 examples the model
has already seen, and paying for it on everything else.

**Now look at the right panel, because it disagrees.** Validation accuracy is
still creeping upward at epoch 40, while validation loss has been getting worse
for thirty epochs. Both are measured on the same 2,000 held-out examples at the
same moment. They cannot both be describing the same thing.

Here is the difference. **Accuracy** asks one question: was the largest logit
the right class? **Cross-entropy** asks a second question on top of that: how
confident were you? As training continues the model gets more confident about
everything. That nudges accuracy up slightly, as borderline cases tip the right
way, and it drives loss up sharply, because being confidently wrong is punished
much harder than being unsure and wrong.

So the two curves are both honest, and they are answering different questions.
The loss is the earlier and more sensitive warning, which is why Part 4 stops
on loss rather than on accuracy. It is also why a model can look like it is
still improving on your accuracy plot long after it has started to go bad.

---

# Part 3 · The optimizer changes the answer

Same model, same seed, same data, same number of epochs. The only thing that
changes is the rule for turning a gradient into a step.

- **`SGD`** moves straight down the gradient. Simple and slow.
- **`SGD(momentum=0.9)`** accumulates a velocity, so consistent directions
  build speed and oscillating ones cancel.
- **`Adam`** keeps a per-parameter running estimate of gradient size and scales
  each step by it, so parameters with small gradients still move.

Build all three and run them. Note that Adam's learning rate is a hundredth of
SGD's; that is normal, and the two numbers are not comparable, because Adam has
already normalized the gradient before the learning rate is applied.

In [ ]:
# TODO: replace each None with a one-argument function that takes the model's
#   parameters and returns a built optimizer, like:
#       lambda p: torch.optim.SGD(p, lr=0.1)
#   Use plain SGD at lr=0.1; SGD at lr=0.1 with momentum=0.9; Adam at lr=1e-3.
OPTIMIZERS = [
    ("SGD",          None),
    ("SGD+momentum", None),
    ("Adam",         None),
]

if any(f is None for _, f in OPTIMIZERS):
    raise NotImplementedError("fill in OPTIMIZERS above")

runs = {}
for name, build_opt in OPTIMIZERS:
    m = make_model().to(DEVICE)
    print(name)
    runs[name] = fit(m, build_opt(m.parameters()), log_every=0)
    b = int(np.argmin(runs[name]["val"]))
    print(f"  best val {runs[name]['val'][b]:.4f} at epoch {b}, "
          f"accuracy {runs[name]['acc'][b]:.4f}, "
          f"{runs[name]['seconds']:.1f}s")

In [ ]:
plot_history(runs, "Three optimizers, everything else held fixed")

print(f"{'optimizer':<15s} {'train end':>10s} {'best val':>10s} "
      f"{'at epoch':>9s} {'best acc':>9s}")
for name, h in runs.items():
    b = int(np.argmin(h["val"]))
    print(f"{name:<15s} {h['train'][-1]:>10.4f} {h['val'][b]:>10.4f} "
          f"{b:>9d} {h['acc'][b]:>9.4f}")

Three things worth noticing.

**Plain SGD is still going down at epoch 40.** Its training loss is the highest
of the three, not because it is worse but because it is slower. Given 200
epochs it would get there.

**Momentum and Adam both reach a low training loss quickly, and both then
overfit.** Faster optimization is not better generalization. The optimizer
decides how fast you reach the bottom of the training loss, and reaching it
sooner just means you start overfitting sooner.

**The best validation losses are close together**, and plain SGD, the slowest
of the three to fit the training data, is right there with the other two. The
optimizer mattered enormously for how fast the training loss fell and very
little for the best validation number any of them reached. That is the normal
outcome on a problem this size, and it is worth knowing before you spend a week
tuning an optimizer.

::: {.callout-tip}
## Learning rate schedules
Dropping the learning rate partway through is usually worth more than switching
optimizers. On this exact setup, cosine annealing over 40 epochs took the
validation accuracy from 0.844 to 0.863 with no other change. One line:

```python
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40)
```

`fit()` already accepts a `scheduler=` argument and calls `.step()` on it each
epoch. Try it if you have time.
:::

---

# Part 4 · Overfitting, early stopping, and the test set

## 4a · Stop at the bottom

Every run in Part 3 hit its best validation loss and then got worse. The fix is
not clever: **keep the weights from the best epoch and stop when they stop
improving.**

Two pieces:

- **Patience.** Validation loss is noisy, so one bad epoch means nothing. Wait
  `patience` epochs without improvement before you quit.
- **Restore.** Stopping is not enough. When you stop, the weights in the model
  are from the *last* epoch, not the best one. You have to have saved a copy.

That copy must be a `.clone()` of each tensor. `model.state_dict()` returns
references to the live tensors, so if you keep it without cloning, training
keeps modifying the thing you thought you saved.

Patience is itself a knob, and the run below prints enough to see the tradeoff.
On the setup in 4c, `patience=8` stops around epoch 18 and keeps epoch 9;
`patience=20` runs to about epoch 60 and keeps epoch 39, which really is the
better epoch. But the test accuracy between those two choices differs by about
two tenths of a percent. The validation curve is nearly flat along the bottom,
so the exact epoch you stop on matters much less than stopping at all.

In [ ]:
def fit_early_stopping(model, optimizer, max_epochs=80, patience=15):
    """Train until validation stops improving, then restore the best weights."""
    loss_fn = nn.CrossEntropyLoss()
    hist = {"train": [], "val": [], "acc": []}
    best_loss, best_epoch, best_state, waited = float("inf"), -1, None, 0
    train_dl.generator.manual_seed(SEED)

    for epoch in range(max_epochs):
        tr = train_one_epoch(model, train_dl, optimizer, loss_fn)
        vl, va = evaluate(model, val_dl)
        hist["train"].append(tr); hist["val"].append(vl); hist["acc"].append(va)

        # TODO: the early stopping bookkeeping.
        #   If vl improved on best_loss: update best_loss and best_epoch,
        #     reset waited to 0, and save a CLONED copy of model.state_dict()
        #     into best_state.
        #   Otherwise: increment waited, and if waited >= patience, print a
        #     message and break out of the loop.
        #   Clone with: {k: v.clone() for k, v in model.state_dict().items()}
        raise NotImplementedError("early stopping")

    model.load_state_dict(best_state)
    hist["best_epoch"] = best_epoch
    return hist


model = make_model().to(DEVICE)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
es = fit_early_stopping(model, optimizer)

print(f"ran {len(es['val'])} epochs, kept epoch {es['best_epoch']}")
print(f"validation at the epoch we kept : {es['val'][es['best_epoch']]:.4f}")
print(f"validation at the epoch we ended: {es['val'][-1]:.4f}")

## 4b · Or make the rise smaller in the first place

Early stopping catches the best epoch. Regularization makes the choice of epoch
matter less, which is worth more than it sounds when you are running things
overnight.

Two levers, both one argument:

- **Dropout** zeroes a random fraction of hidden units on each training batch,
  so no single unit can be relied on. `nn.Dropout(0.3)`, which `make_model`
  already takes.
- **Weight decay** adds a penalty on the size of the weights, pulling them
  toward zero unless the data pushes back. `weight_decay=1e-3` on the
  optimizer.

Run all four combinations and compare how far the validation loss climbs after
its minimum.

In [ ]:
reg_runs = {}
for name, p_drop, wd in [("plain", 0.0, 0.0),
                         ("dropout 0.3", 0.3, 0.0),
                         ("weight decay 1e-3", 0.0, 1e-3),
                         ("both", 0.3, 1e-3)]:
    m = make_model(p_drop=p_drop).to(DEVICE)
    opt = torch.optim.SGD(m.parameters(), lr=0.1, momentum=0.9, weight_decay=wd)
    reg_runs[name] = fit(m, opt, log_every=0)

plot_history(reg_runs, "Same optimizer, four regularization settings")

print(f"{'setting':<20s} {'train end':>10s} {'best val':>10s} {'val end':>9s} "
      f"{'rise':>7s} {'best acc':>9s}")
for name, h in reg_runs.items():
    b = int(np.argmin(h["val"]))
    print(f"{name:<20s} {h['train'][-1]:>10.4f} {h['val'][b]:>10.4f} "
          f"{h['val'][-1]:>9.4f} {h['val'][-1] - h['val'][b]:>+7.3f} "
          f"{h['acc'][b]:>9.4f}")

**Read the `rise` column first.** That is how much worse validation got between
its best epoch and epoch 40, and it is the number these two levers are supposed
to move.

They do not move it equally, and the result is not the tidy one you might
expect. **Dropout flattens the curve hard**: on its own it cuts the climb to a
fraction of the plain run's, and combined with weight decay the curve is
essentially flat, meaning that model sits at its best for the entire back half
of training. **Weight decay on its own does not help here at all**, and on this
particular run it climbs further than the plain model does.

That is worth sitting with rather than explaining away. Weight decay is a real
and widely used regularizer, but at `1e-3` on a 200,000-parameter model trained
for 40 epochs with momentum, it was not the binding constraint. A different
strength, or a longer run, would give a different answer. **Regularizers are
not additive goods you switch on; each one is a setting you have to tune, and
some of them do nothing on your problem.**

**Now read the `best val` column, which barely moves at all.** Across all four
settings it spans a couple of hundredths. So regularization did not really make
the best achievable number better. It made the run **stable**, which is a
different and more practical thing: you no longer have to catch one specific
epoch to get a good model.

That is worth having precisely because early stopping is imperfect. If you
could always stop on exactly the right epoch you would not need this. Look at
how wide the plain run's rise is and imagine stopping ten epochs late.

## 4c · The test set, once

You have now looked at validation loss many times: to pick an optimizer, to
pick a stopping epoch, to pick a regularizer. Every one of those decisions fit
a little bit to the validation set, which is why validation is now slightly
optimistic about your model.

The test set has been untouched since 1c. Touch it once, report the number, and
do not go back and change anything based on it. If you tune after looking, you
have converted your test set into a second validation set and you no longer
have an honest estimate.

In [ ]:
final_model = make_model(p_drop=0.3).to(DEVICE)
final_opt = torch.optim.SGD(final_model.parameters(), lr=0.1, momentum=0.9,
                            weight_decay=1e-3)
final_hist = fit_early_stopping(final_model, final_opt)

val_loss, val_acc = evaluate(final_model, val_dl)
test_loss, test_acc = evaluate(final_model, test_dl)

print(f"\nkept epoch {final_hist['best_epoch']} of {len(final_hist['val'])}")
print(f"  validation : loss {val_loss:.4f}   accuracy {val_acc:.4f}   "
      f"({len(val_ds):,} examples, looked at every epoch)")
print(f"  TEST       : loss {test_loss:.4f}   accuracy {test_acc:.4f}   "
      f"({len(test_ds):,} examples, looked at once)")
print(f"\ntrained on {TRAIN_N:,} of 60,000 available examples")

In [ ]:
# Where the remaining errors are. The confusion is not random, and next week's
# lecture is about the structure this model is throwing away.
final_model.eval()
confusion = torch.zeros(10, 10, dtype=torch.int32)
with torch.no_grad():
    for xb, yb in test_dl:
        pred = final_model(xb.to(DEVICE)).argmax(dim=1).cpu()
        for t, p in zip(yb, pred):
            confusion[t, p] += 1

per_class = confusion.diag().float() / confusion.sum(dim=1).float()
order = torch.argsort(per_class)

print("hardest classes for this model:")
for i in order[:4]:
    wrong = confusion[i].clone(); wrong[i] = 0
    worst = int(wrong.argmax())
    print(f"  {CLASSES[i]:<12s} {per_class[i]:.3f} correct, "
          f"most often called {CLASSES[worst]!r}")
print("\neasiest:")
for i in order.flip(0)[:3]:
    print(f"  {CLASSES[i]:<12s} {per_class[i]:.3f} correct")

The mistakes are not spread evenly. Look at which classes are hard: they are
the upper-body garments, and they get confused with each other. Shirts,
T-shirts, pullovers and coats are genuinely similar at 28 by 28 pixels once you
have flattened them into a vector of 784 numbers. Bags, trousers and boots have
distinctive outlines and the model gets them nearly always.

And note what "flattened" cost us. `__getitem__` called `.view(-1)`, which threw
away the fact that pixel 100 sits directly above pixel 128. The model had to
learn spatial structure from scratch, from 6,000 examples, with no hint that
the image was ever two-dimensional. **That is what convolutional networks fix,
and that is next week.**

---

# Part 5 · What this lab showed

**This is the part that gets assessed**, on Quiz 5. Nothing to memorize and no
numbers to transcribe: the quiz asks *why*, not *what was your value*.

Scroll back through your own output and make sure you can say, in a sentence
each:

1. **What `__getitem__` is for**, and why the normalization and the reshape
   live there rather than in `__init__`.
2. **When `num_workers` helps and when it hurts.** You measured both. What is
   the property of the `Dataset` that decides which one you get?
3. **Why the validation set is carved out of train and not out of test**, and
   what specifically goes wrong if you tune against test.
4. **Why the last layer has no softmax**, and what `CrossEntropyLoss` is doing
   that makes that the right choice.
5. **What `optimizer.zero_grad()` prevents**, and why PyTorch accumulates
   gradients by default instead of overwriting them.
6. **What `model.eval()` and `torch.no_grad()` each do**, and which one you
   would notice missing.
7. **Why you overfit 100 examples before starting a real run**, and what a
   failure at that step rules out.
8. **What the gap between the training and validation curves means**, and why
   the training loss continuing to fall is not good news.
9. **How validation accuracy can still be rising while validation loss is
   getting worse.** You saw this happen in 2e. Which of the two would you stop
   on, and why?
10. **Why early stopping needs a cloned copy of the weights** rather than just
    a stopping rule.
11. **Why regularization barely moved the best validation loss but clearly
    flattened the curve after it**, and why that is still worth having.

If you can answer those eleven, you are ready for the quiz.

---

# Part 6 · AI disclosure

Required if you used any generative AI on this lab. Name the tool, say where you
used it, and say what for. "None" is a perfectly good answer.

*Tool(s):*

*Where:*

*What for:*

---

# Submitting

1. **Restart the kernel and Run All.** Confirm it runs top to bottom.
2. Check that every plot and every printed block is visible.
3. Render to **HTML or PDF** with resources embedded.
4. Submit the rendered file on Canvas. Due **Wed Sep 30, 11:59 PM ET**.

The notebook is graded for completion, so this is mostly a formality. Part 5 is
your study guide for the quiz that assesses the understanding.

::: {.callout-note}
## Do not submit the data folder
Running this notebook creates a `data/` directory of about 86 MB next to it.
Submit the rendered HTML or PDF, not the folder.
:::